In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# to have interactive plots : pip install ipympl
%matplotlib ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import emcee
import pickle
import os

from qubic.lib import Qdictionary, Qinstrument
from qubic.lib.Instrument.Qinstrument import QubicMultibandInstrument
from qubic.lib.Qscene import QubicScene
from qubicpack.utilities import Qubic_DataDir

import pipeline_moon_functions as pmf
import pipeline_moon_plotting as pmp

## We choose the instrument configuration (will probably be TD for a while)

This is to be updated with the current version of the soft

In [ ]:
### Let's simulate TOD from this
# Repository for dictionary and input maps
global_dir = Qubic_DataDir(datafile='instrument.py', datadir='../')
dictfilename = global_dir + '/dicts/pipeline_MoonSalta.dict'
d = Qdictionary.qubicDict()
d.read_from_file(dictfilename)
d['nside'] = 512 #256
d['config'] = 'TD' # no calfile for TD, or even for FI at 150 GHz
d['kind'] = 'I'
d['nf_sub'] = 12 # monoband not implemented on this branch
d['nf_recon'] = 2
d['MultiBand'] = True
d['sweeping_pointing_deadtime'] = False
d['noiseless'] = True
d["use_synthbeam_fits_file"] = False
d["n_nu_fit"] = d['nf_sub'] # monoband not implemented on this branch, but seems like all nsub are in the right band

d['detector_tau'] = 0.01

In [ ]:
ang_size = 40 # degrees
xsize = 201
reso = ang_size*60/xsize # arcmin/pix
min_plot = -5e3
max_plot = 1.2e4
zenith = [0, 90]

In [ ]:
print(reso)

## We read the maps

In [ ]:
datadir_MoonProject = "/Users/huchet/Documents/code/scripts/MoonProject/"

In [ ]:
det_los = True # we want the maps centered at each detector los
nside = 512
boresight_angle = -3.2

more_name = "202607_stack"
title = "Stacked maps (July 2026)"
if det_los:
    more_name += "_detzenith"
    title += "det l.o.s. at zenith"

if boresight_angle == -3.5:
    file_name = "allmaps-{}_nside{}_zenith.pkl".format(more_name, nside)
else:
    file_name = "allmaps-{}_nside{}_boresight{}_zenith.pkl".format(more_name, nside, boresight_angle)

allTESNum, stacked_maps, chosen_patch = pickle.load( open( datadir_MoonProject + file_name, "rb" ) )
print("Read {}".format(file_name))

remap = pmf.get_remapping_operator(chosen_patch, np.shape(chosen_patch), (12*nside**2))

## We will now fit the Moon spectrum

Ideas tested so far:
- least_squares: doesn't fit right, probably too many d.o.f.
- curve_fit: same
- emcee on each spectrum value: same, have to reduce the number of nus a lot for it to work
- emcee on linear spectrum: best approach yet, can keep all nus and fit only 2 params

emcee on linear spectrum works for a flat spectrum and is OK with non-flat linear spectrum.

In [ ]:
TESNum = 96
idet_qp = TESNum - 1

In [ ]:
map_TES = remap(stacked_maps[idet_qp])

In [ ]:
print(np.shape(stacked_maps))
print(np.shape(map_TES))

In [ ]:
moon_angsize = np.radians(0.5) # radians
zenith = np.array([0, 90, 0])

# in case we use a "flat" Moon
def top_hat_kernel(b, radius): # we create a new kernel for the convolution
    return np.where(np.abs(b) <= radius, 1, 0)
b = np.linspace(0, np.pi, 10000)
kernel_real = top_hat_kernel(b, moon_angsize)
nside = 512
beam_window = hp.sphtfunc.beam2bl(kernel_real, b, nside * 3) # need a better sampling than the one used in the maps

In [ ]:
def get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix, reso):
    idet_qs = pmf.iQP2iQS(idet_qp)
    i_peak = 4 # order 0 peak in the case of Kmax = 1
    n_nus = len(q_instrument)
    sb_proj_nus = np.zeros((Npix, Npix, n_nus))
    for i_nu in range(n_nus):
        # we get the shape of the synthbeam
        sb_test = q_instrument[i_nu].get_synthbeam(q_scene, idet_qs, theta_max)
        # if the Moon is Gaussian
        # sb_test_moon = hp.smoothing(sb_test, fwhm=moon_angsize)
        # if the Moon is "flat"
        sb_test_moon = hp.smoothing(sb_test, beam_window=beam_window)
        # we get the position of the peaks
        thetas, phis, _ = Qinstrument.QubicInstrument._peak_angles_unsorted(q_scene, q_instrument[i_nu].filter.nu,
            q_instrument[i_nu].detector.center, q_instrument[i_nu].synthbeam, q_instrument[i_nu].horn, q_instrument[i_nu].primary_beam)
        els_deg = 90 - np.degrees(thetas)
        azs_deg = np.degrees(phis)
        det_pos = (azs_deg[idet_qs, i_peak], els_deg[idet_qs, i_peak], -azs_deg[idet_qs, i_peak]) # we put the order 0 peak at the centre
        # we swap the axes to fit what we see in the data!!
        # might be the same thing as the detector line of sights computed by Maynooth
        # sb_proj_nus[:, :, i_nu] = np.swapaxes(hp.gnomview(sb_test_moon, rot=det_pos, reso=reso, xsize=Npix, return_projected_map=True, no_plot=True).data, 0, 1)
        # plan B, we first rotate at zenith and then project from there in order to have the same pixel shape
        test_rot = [det_pos[0], det_pos[1] - 90, det_pos[2]]
        # print(det_pos)
        # print(test_rot)
        # we rotate the sunthbeam from zenith to the observed detector position
        rotator = hp.rotator.Rotator(rot=test_rot, coord=None, inv=None, deg=True, eulertype='Y')
        sb_test_moon_rot = rotator.rotate_map_alms(sb_test_moon)
        sb_proj_nus[:, :, i_nu] = np.swapaxes(hp.gnomview(sb_test_moon_rot, rot=zenith, reso=reso, xsize=Npix, return_projected_map=True, no_plot=True).data, 0, 1)
        # plt.figure()
        # hp.gnomview(sb_test_moon, rot=zenith, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 1))
        # # hp.gnomview(sb_test_moon, rot=det_pos, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 2))
        # plt.show()
    return sb_proj_nus

def get_sb_nus_allTES(q_scene, q_instrument, TES_list, theta_max, Npix, reso):
    i_peak = 4 # order 0 peak in the case of Kmax = 1
    n_nus = len(q_instrument)
    n_TES = len(TES_list)
    sb_proj_nus_allTES = np.zeros((Npix, Npix, n_TES, n_nus)) # I think the dimensions have to be ranked by size (decreasing for C-order?)
    for i_nu in range(n_nus):
        # we get the shape of the synthbeam
        for i_TES, TESNum_qp in enumerate(TES_list):
            idet_qs = pmf.iQP2iQS(TESNum_qp - 1)
            sb_test = q_instrument[i_nu].get_synthbeam(q_scene, idet_qs, theta_max)
            # if the Moon is Gaussian
            # sb_test_moon = hp.smoothing(sb_test, fwhm=moon_angsize)
            # if the Moon is "flat" (seems to fit the data better?)
            sb_test_moon = hp.smoothing(sb_test, beam_window=beam_window)
            # we get the position of the peaks
            thetas, phis, _ = Qinstrument.QubicInstrument._peak_angles_unsorted(q_scene, q_instrument[i_nu].filter.nu,
                q_instrument[i_nu].detector.center, q_instrument[i_nu].synthbeam, q_instrument[i_nu].horn, q_instrument[i_nu].primary_beam)
            els_deg = 90 - np.degrees(thetas)
            azs_deg = np.degrees(phis)
            det_pos = (azs_deg[idet_qs, i_peak], els_deg[idet_qs, i_peak], -azs_deg[idet_qs, i_peak]) # we put the order 0 peak at the centre
            # we swap the axes to fit what we see in the data!!
            # might be the same thing as the detector line of sights computed by Maynooth
            # we first rotate at zenith and then project from there in order to have the same pixel shape
            test_rot = [det_pos[0], det_pos[1] - 90, det_pos[2]]
            # print(det_pos)
            # print(test_rot)
            # we rotate the sunthbeam from zenith to the observed detector position
            rotator = hp.rotator.Rotator(rot=test_rot, coord=None, inv=None, deg=True, eulertype='Y')
            sb_test_moon_rot = rotator.rotate_map_alms(sb_test_moon)
            sb_proj_nus_allTES[:, :, i_TES, i_nu] = np.swapaxes(hp.gnomview(sb_test_moon_rot, rot=zenith, reso=reso, xsize=Npix, return_projected_map=True, no_plot=True).data, 0, 1)
            # plt.figure()
            # hp.gnomview(sb_test_moon, rot=zenith, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 1))
            # # hp.gnomview(sb_test_moon, rot=det_pos, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 2))
            # plt.show()
    return sb_proj_nus_allTES

In [ ]:
q_scene = QubicScene(d)
q_instrument = QubicMultibandInstrument(d)

n_nus = len(q_instrument)
nus = np.zeros(n_nus)
dnus = np.zeros(n_nus)
print()
print('The instrument has {} sub-frequencies'.format(n_nus))
for i in range(n_nus):
    nus[i] = q_instrument.subinstruments[i].d['filter_nu']/1e9
    dnus[i] = q_instrument.subinstruments[i].d['filter_relative_bandwidth']*q_instrument.subinstruments[i].d['filter_nu']/1e9

In [ ]:
# we get the sb

theta_max = 25
reso_test = 4 #2
Npix_test = 300 #600
X = np.arange(Npix_test)
Y = np.arange(Npix_test)
# XX, YY = np.meshgrid(X, Y, indexing="xy") # pixel units, just for the interpolation
sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix_test, reso_test)

In [ ]:
# Visualisation on one TES


rot_test = [0, 90, -0.8] # rotation of -1 deg
map_proj = hp.gnomview(map_TES, rot=rot_test, reso=reso_test, xsize=Npix_test, return_projected_map=True, no_plot=True)
no_UNSEEN_mask = ~map_proj.mask
# interpolator = LinearNDInterpolator(np.moveaxis([XX[no_UNSEEN_mask], YY[no_UNSEEN_mask]], 0, -1), map_proj[no_UNSEEN_mask])
# map_proj = interpolator(np.moveaxis([XX, YY], 0, -1))

# sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix, reso)
# map_proj = allmaps_ok_proj[np.cumsum(visibly_ok_arr)[idet_qp] - 1]
map_proj[~np.isfinite(map_proj)] = 0
map_proj[map_proj == hp.UNSEEN] = 0
map_proj[map_proj.mask] = 0
# map_proj = map_proj/amp_TES * all_simu_amp
fig, axs = plt.subplots(1, 2, figsize = (15, 7))
axs[0].set_title("sb_proj_nus")
axs[0].imshow(np.mean(sb_proj_nus, axis=-1))
axs[1].set_title("map_proj")
axs[1].imshow(map_proj)
plt.show()

In [ ]:
# zergin

In [ ]:
# emcee approach

def fit_fun_moon_spectrum(lin_spectrum_params, sb_proj_nus, map_proj, nus): # I should probably choose one convention (iTES being first or last axis always)
    Npix = int(np.sqrt(len(map_proj)))
    # print(Npix, len(map_proj))
    # n_nus = len(sb_proj_nus)//Npix**2
    n_nus = len(nus)
    weights = lin_spectrum_params[0] * nus + lin_spectrum_params[1] # should be value of nus and not arange here
    map_proj = map_proj.reshape((Npix, Npix))
    sb_proj_nus = sb_proj_nus.reshape((Npix, Npix, n_nus))
    # the difference betweent he maps should be done only neaar the peaks, need to devise the mask for that and input it into the function
    # let's use the synthbeam values as mask and also a mask describing the pixels seen
    mask_seen = map_proj != 0
    # we stop caring about the centre to try to reduce the dof
    # XX, YY = np.meshgrid(np.arange(Npix), np.arange(Npix), indexing="xy")
    # mask_close_cent = np.sqrt((XX - Npix//2)**2 + (YY - Npix//2)**2) > 20 # 20 pixels might not be enough?
    tot_mask = mask_seen#*mask_close_cent
    # fig, axs = plt.subplots(1, 2)
    # axs[0].imshow(tot_mask*sb_proj_nus[:, :, 0])
    # axs[1].imshow(map_proj)
    # plt.show()
    # azet
    return -np.sum(np.abs(tot_mask*(map_proj - np.sum(weights[None, None, :] * sb_proj_nus, axis=-1))))

def fit_fun_moon_spectrum_allTES(lin_spectrum_params, sb_proj_nus_allTES, all_maps, nus, list_TES): # I should probably choose one convention (iTES being first or last axis always)

    n_dets = len(list_TES)
    Npix = int(np.sqrt(len(all_maps)/n_dets))
    # print(Npix, len(all_maps))
    # n_nus = len(sb_proj_nus)//Npix**2
    n_nus = len(nus)
    weights = lin_spectrum_params[0] * nus + lin_spectrum_params[1] # should be value of nus and not arange here
    all_maps = all_maps.reshape((Npix, Npix, n_dets))
    sb_proj_nus_allTES = sb_proj_nus_allTES.reshape((Npix, Npix, n_dets, n_nus))
    # the difference betweent he maps should be done only neaar the peaks, need to devise the mask for that and input it into the function
    # let's use the synthbeam values as mask and also a mask describing the pixels seen
    mask_seen = all_maps != 0
    # we stop caring about the centre to try to reduce the dof
    # XX, YY = np.meshgrid(np.arange(Npix), np.arange(Npix), indexing="xy")
    # mask_close_cent = np.sqrt((XX - Npix//2)**2 + (YY - Npix//2)**2) > 20 # 20 pixels might not be enough?
    tot_mask = mask_seen#*mask_close_cent
    # fig, axs = plt.subplots(1, 2)
    # axs[0].imshow(tot_mask*sb_proj_nus_allTES[:, :, 0])
    # axs[1].imshow(all_maps)
    # plt.show()
    # azet
    return -np.sum(np.abs(tot_mask*(all_maps - np.sum(weights[None, None, None, :] * sb_proj_nus_allTES, axis=-1))))

# def log_prob(spectrum_weights, others):
#     if np.any(spectrum_weights<0):
#         return -np.inf
#     sb_proj_nus, map_proj, nus = others
#     return np.log(fit_fun_moon_spectrum(spectrum_weights, sb_proj_nus, map_proj, nus)/)

# fit_fun_moon_spectrum(moon_spectrum_true, sb_proj_nus, map_proj.ravel(), nus)
# azef




In [ ]:
# step1_pos = np.load("full_det_pos_zen.npy")
# step1_fit_maynooth = np.load("fitted_maynooth.npy")
# DBscan_ok_step1 = (step1_pos == step1_fit_maynooth)
# DBscan_ok_step1 = np.prod(~DBscan_ok_step1 & np.isfinite(step1_pos), axis=1, dtype=bool)

# bad_VI = np.array([14])
# DBscan_ok_step1[bad_VI - 1] = False # by eye

# all_TES_ok = allTESNum[DBscan_ok_step1]

best_VI = np.array([34, 69, 70, 76, 83, 93, 94, 95, 96, 99, 131, 140, 172, 193, 194, 207, 218])
all_TES_ok = best_VI

all_maps = []

for TESNum_ in all_TES_ok:
    idet_qp = TESNum_ - 1
    map_TES_ = remap(stacked_maps[idet_qp])
    map_proj = hp.gnomview(map_TES_, rot=rot_test, reso=reso_test, xsize=Npix_test, return_projected_map=True, no_plot=True)
    no_UNSEEN_mask = ~map_proj.mask
    map_proj[~np.isfinite(map_proj)] = 0
    map_proj[map_proj == hp.UNSEEN] = 0
    map_proj[map_proj.mask] = 0
    all_maps.append(map_proj)

all_maps = np.array(all_maps)

In [ ]:
# need to get theo_sb for all TES (too long to run)
# sb_proj_nus_allTES = get_sb_nus_allTES(q_scene, q_instrument, all_TES_ok, theta_max, Npix_test, reso)

In [ ]:
for i_TES, TESNum_ in enumerate(all_TES_ok):
    idet_qp = TESNum_ - 1
    # we get the sb
    # fig, axs = plt.subplots(1, 2)
    fig, ax = plt.subplots()
    fig.suptitle("TES {}".format(TESNum_))
    # sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix_test, reso_test)
    map_proj = all_maps[i_TES]
    ax.imshow(map_proj, vmin=-5e3, vmax=3e4)
    # axs[0].imshow(map_proj)
    # axs[1].imshow(sb_proj_nus[:, :, 0])
    plt.show()

In [ ]:
print(np.shape(all_maps))

In [ ]:
# 2 dims, we fit a line

theta_max = 25
reso_test = 4 #2
Npix_test = 300 #600
for i_TES, TESNum_ in enumerate(all_TES_ok):
    idet_qp = TESNum_ - 1
    # Set up the backend: it will save the chain
    # Don't forget to clear it in case the file already exists
    file_chain = "chain_moon_spectrum_{}_TES{}.h5".format(more_name, TESNum_)
    if os.path.isfile("chains/" + file_chain):
        print("Chain {} already computed.".format(file_chain))
        continue
    print("Working on chain {}".format(file_chain))
    # !rm {file_chain}
    
    # we get the sb
    sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix_test, reso_test)
    map_proj = all_maps[i_TES]
    # we initialise the MCMC
    ndim, nwalkers = 2, 100
    p0 = np.random.rand(nwalkers, ndim) * 10
    # print(p0, np.shape(p0))

    backend = emcee.backends.HDFBackend("chains/" + file_chain)
    backend.reset(nwalkers, ndim)
    # Initialize the sampler
    # one TES
    sampler = emcee.EnsembleSampler(nwalkers, ndim, fit_fun_moon_spectrum, args=[sb_proj_nus.ravel(), map_proj.ravel(), nus], backend=backend)
    sampler.run_mcmc(p0, 5000)
    print("MCMC finished.")
    # aer
# all TES in one go
# sampler = emcee.EnsembleSampler(nwalkers, ndim, fit_fun_moon_spectrum_allTES, args=[sb_proj_nus_allTES.ravel(), all_maps.ravel(), nus, all_TES_ok], backend=backend)

In [ ]:
# We now study the chains

plt.figure()
plt.title("spectrum vs nu")

for i_TES, TESNum_ in enumerate(all_TES_ok):
    idet_qp = TESNum_ - 1
    # Set up the backend: it will save the chain
    # Don't forget to clear it in case the file already exists
    file_chain = "chain_moon_spectrum_{}_TES{}.h5".format(more_name, TESNum_)
    if os.path.isfile("chains/" + file_chain):
        print("Reading chain {}.".format(file_chain))
    else:
        continue
    reader = emcee.backends.HDFBackend("chains/" + file_chain)

    tau = reader.get_autocorr_time()
    burnin = int(2 * np.max(tau))
    thin = int(0.5 * np.min(tau))
    samples = reader.get_chain(discard=burnin, flat=True, thin=thin)
    log_prob_samples = reader.get_log_prob(discard=burnin, flat=True, thin=thin)

    flat_samples = reader.get_chain(discard=100, thin=15, flat=True)
    print(flat_samples.shape)

    labels = ["a", "b"]
    fitted_lin_spectrum_params = []
    for i in range(ndim):
        mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])
        q = np.diff(mcmc)
        fitted_lin_spectrum_params.append(mcmc[1])
        # txt = "\mathrm{{{3}}} = {0:.3f}_{{-{1:.3f}}}^{{{2:.3f}}}"
        # txt = txt.format(mcmc[1], q[0], q[1], labels[i])
        # print(txt)
        print("{} = {} + {} - {}".format(labels[i], mcmc[1], q[0], q[1]))
    fitted_spectrum = fitted_lin_spectrum_params[0] * nus + fitted_lin_spectrum_params[1]
    fitted_map = no_UNSEEN_mask*np.sum(fitted_spectrum[None, None, :] * sb_proj_nus, axis=-1)
    plt.plot(nus, fitted_spectrum, label="{}".format(TESNum_))
    
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
# %%script echo one chain case
# We now study the chains

TESNum = 15
file_chain = "chain_moon_spectrum_{}_TES{}.h5".format(more_name, TESNum)
reader = emcee.backends.HDFBackend("chains/" + file_chain)

tau = reader.get_autocorr_time()
burnin = int(2 * np.max(tau))
thin = int(0.5 * np.min(tau))
samples = reader.get_chain(discard=burnin, flat=True, thin=thin)
log_prob_samples = reader.get_log_prob(discard=burnin, flat=True, thin=thin)
# log_prior_samples = reader.get_blobs(discard=burnin, flat=True, thin=thin)

print("burn-in: {0}".format(burnin))
print("thin: {0}".format(thin))
print("flat chain shape: {0}".format(samples.shape))
print("flat log prob shape: {0}".format(log_prob_samples.shape))
# print("flat log prior shape: {0}".format(log_prior_samples.shape))

In [ ]:
flat_samples = reader.get_chain(discard=100, thin=15, flat=True)
print(flat_samples.shape)

labels = ["a", "b"]
fitted_lin_spectrum_params = []
for i in range(ndim):
    mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    fitted_lin_spectrum_params.append(mcmc[1])
    # txt = "\mathrm{{{3}}} = {0:.3f}_{{-{1:.3f}}}^{{{2:.3f}}}"
    # txt = txt.format(mcmc[1], q[0], q[1], labels[i])
    # print(txt)
    print("{} = {} + {} - {}".format(labels[i], mcmc[1], q[0], q[1]))
fitted_spectrum = fitted_lin_spectrum_params[0] * nus + fitted_lin_spectrum_params[1]

In [ ]:
fitted_map = no_UNSEEN_mask*np.sum(fitted_spectrum[None, None, :] * sb_proj_nus, axis=-1)

plt.figure()
plt.title("spectrum vs nu")
plt.plot(nus, fitted_spectrum, label="fitted")
plt.yscale("log")
plt.legend()
plt.show()

iTES = int(np.argwhere(all_TES_ok == TESNum))
map_proj = all_maps[iTES]

vmin, vmax = np.min(map_proj), np.max(map_proj)
ncols = 4
fig, axs = plt.subplots(1, ncols, figsize=(ncols*4, 5))
axs[0].set_title("map_proj")
axs[0].imshow(map_proj, vmin=vmin, vmax=vmax)
axs[1].set_title("fitted_map")
axs[1].imshow(fitted_map, vmin=vmin, vmax=vmax)
axs[2].set_title("map_proj - fitted_map")
axs[2].imshow(map_proj - fitted_map, vmin=vmin, vmax=vmax)
axs[3].set_title("abs(map_proj - fitted_map)/abs(map_proj)")
plot_rel_diff = np.abs(map_proj - fitted_map)/np.abs(map_proj)
condition = np.abs(map_proj - fitted_map)<1e1
plot_rel_diff[condition] = 0
axs[3].imshow(plot_rel_diff, vmin=0, vmax=5e-1)
plt.show()

plt.figure()
plt.imshow(condition)
plt.show()